# Bloc 02 - Speed Dating 

## 02 - Nettoyage des données

Import de librairies

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("../data/data.csv")

In [3]:
with pd.option_context(
    "display.max_columns", None, 
    "display.max_rows", None,):
    pourcentage_na = data.isna().mean() * 100
    print(f"Pourcentage de données manquantes pour chaque colonne :\n{pourcentage_na}")

Pourcentage de données manquantes pour chaque colonne :
Unnamed: 0      0.000000
iid             0.000000
id              0.011936
gender          0.000000
idg             0.000000
condtn          0.000000
wave            0.000000
round           0.000000
position        0.000000
positin1       22.033898
order           0.000000
partner         0.000000
pid             0.119360
match           0.000000
int_corr        1.885892
samerace        0.000000
age_o           1.241346
race_o          0.871330
pf_o_att        1.062306
pf_o_sin        1.062306
pf_o_int        1.062306
pf_o_fun        1.169730
pf_o_amb        1.277154
pf_o_sha        1.539747
dec_o           0.000000
attr_o          2.530437
sinc_o          3.425639
intel_o         3.652423
fun_o           4.296968
amb_o           8.617809
shar_o         12.843161
like_o          2.984006
prob_o          3.795655
met_o           4.595369
age             1.133922
field           0.751969
field_cd        0.978754
undergra       41.3

### a) Id: donnée manquante

In [4]:
# id           0.011936
print(data[data["id"].isna()]["iid"])

# un oubli l'id est 22
data.loc[8377, "id"] = 22

8377    552
Name: iid, dtype: int64


### b) Positin1: Suppression de colonnes

In [5]:
# positin1    22.033898
print(data[~data["positin1"].isna()]["wave"].unique())

# existant a partir de la wave 6
# on peut supprimer cette colonne l'information peut etre retrouver avec la colonne order
data.drop(columns="positin1", inplace=True)

[ 6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21]


### c) Pid: suppression de lignes

In [6]:
# pid          0.119360
pid_null = data[data["pid"].isna()]
print(pid_null[["iid", "gender", "wave", "position", "partner"]])

# pendant la wave 5, 
# 10 hommes ont rencontre la partenaire 7 
# or celle ci n'existe plus dans le fichier de données
# id = 7, gender = 0, wave = 5

# on peut supprimer les lignes dont le pid est vide
indexes = pid_null.index
data.drop(indexes, inplace=True)

      iid gender  wave  position  partner
1755  122   Male     5         4        7
1765  123   Male     5         4        7
1775  124   Male     5         4        7
1785  125   Male     5         4        7
1795  126   Male     5         4        7
1805  127   Male     5         4        7
1815  128   Male     5         4        7
1825  129   Male     5         4        7
1835  130   Male     5         4        7
1845  131   Male     5         4        7


### d) Suppression de colonnes
Ces colonnes ont ete ecartées de l'analyse car elles présentent un pourcentage de données manquantes trop élevé, rendant leur utilisation statistiquement peu fiable : par exemple undergrd, mn_sat, tuition.

In [7]:
# undergra    41.346383
# mn_sat      62.604440
# tuition     57.233230
data.drop(columns=["undergra", "mn_sat", "tuition"], inplace=True)

# zipcode     12.699928
# income      48.925758
data.drop(columns=["zipcode", "income"], inplace=True)

# expnum      78.515159
data.drop(columns=["expnum"], inplace=True)

### e) Opérations sur données manquantes sur les codes

In [8]:
# field_cd
# Operations Research --> 5
data.loc[data[data["iid"] == 40].index, "field_cd"] = 5

# career_c
# law --> 1
data.loc[data[data["iid"].isin([1,2])].index, "career_c"] = 1

# economist --> 7
data.loc[data[data["iid"].isin([3])].index, "career_c"] = 7

### f) Analyse des données manquantes

In [9]:
def get_personnes_without_data(field, message, id_field="iid"):
    personnes_without_data = data[data[field].isna()][id_field].astype(int).unique()
    print(f"{message} : {personnes_without_data}")

In [10]:
# int_corr     1.885892
# calcule en fonction des centres d'interet
# colonnes : sports, tvsports, exercise, dining, museums, art, hiking, gaming, clubbing, reading, tv, theater, movies, concerts, music, shopping, yoga
# si les centres d'interet n'ont pas ete renseigne, int_corr est vide
#data[data["int_corr"].isna()]
get_personnes_without_data("sports", "Personnes n'ayant pas renseignees ses centres d'interet")

# age_o        1.241346
# variable en fonction de la variable age
# il s'agit de personnes n'ayant egalement pas renseigne leurs centres d'interet
get_personnes_without_data("age_o", "Partenaires n'ayant pas renseignees son age", "pid")

# race_o       0.871330
# variable en fonction de la variable race
# il s'agit de personnes n'ayant egalement pas renseigne leurs centres d'interet
get_personnes_without_data("race_o", "Partenaires n'ayant pas renseignees sa race", "pid")

# pf_o_att     1.062306
# il s'agit de personnes n'ayant egalement pas renseigne leurs centres d'interet
get_personnes_without_data("pf_o_att", "Partenaires n'ayant pas renseignees interet pour le physique", "pid")

## Scorecard
# attr_o       2.530437
# sinc_o       3.425639
# intel_o      3.652423
# fun_o        4.296968
# amb_o        8.617809
# shar_o      12.843161
# il s'agit de l'avis des partenaires sur le sujet de la ligne etudie

# age          1.133922
get_personnes_without_data("age", "Personnes n'ayant pas renseignees son age")

# field        0.751969
get_personnes_without_data("field", "Personnes n'ayant pas renseignees son domaine d'etude")

# race         0.751969
get_personnes_without_data("race", "Personnes n'ayant pas renseignees sa race")

# imprace      0.942946
get_personnes_without_data("imprace", "Personnes n'ayant pas renseignees l'importance de la race")

# imprelig     0.942946
get_personnes_without_data("imprelig", "Personnes n'ayant pas renseignees l'importance de la religion")

# from         0.942946
get_personnes_without_data("from", "Personnes n'ayant pas renseignees son precedent lieu de residence")

# goal         0.942946
get_personnes_without_data("goal", "Personnes n'ayant pas renseignees le but du speed dating")

# date         1.157794
get_personnes_without_data("date", "Personnes n'ayant pas renseignees de frequence de date")

# go_out       0.942946
get_personnes_without_data("go_out", "Personnes n'ayant pas renseignees de frequence de sortie")

# career       1.062306
get_personnes_without_data("career", "Personnes n'ayant pas renseignees de carriere")

# career_c     1.647171
get_personnes_without_data("career_c", "Personnes n'ayant pas renseignees de code de carriere")

# exphappy     1.205538
get_personnes_without_data("exphappy", "Personnes n'ayant pas renseignees de prevision d'etre avec une personne rencontree")

Personnes n'ayant pas renseignees ses centres d'interet : [ 28  58  59 136 339 340 346]
Partenaires n'ayant pas renseignees son age : [ 58  59 129 136 339 340 346 512]
Partenaires n'ayant pas renseignees sa race : [ 58  59 136 339 340 346]
Partenaires n'ayant pas renseignees interet pour le physique : [ 28  58  59 136 339 340 346]
Personnes n'ayant pas renseignees son age : [ 58  59 129 136 339 340 346 512]
Personnes n'ayant pas renseignees son domaine d'etude : [ 58  59 136 339 340 346]
Personnes n'ayant pas renseignees sa race : [ 58  59 136 339 340 346]
Personnes n'ayant pas renseignees l'importance de la race : [ 28  58  59 136 339 340 346]
Personnes n'ayant pas renseignees l'importance de la religion : [ 28  58  59 136 339 340 346]
Personnes n'ayant pas renseignees son precedent lieu de residence : [ 28  58  59 136 339 340 346]
Personnes n'ayant pas renseignees le but du speed dating : [ 28  58  59 136 339 340 346]
Personnes n'ayant pas renseignees de frequence de date : [ 28  58 

### g) Suppression de personnes  
Comme identifié lors de l'analyse des données manquantes, les 7 participants suivants (28, 58, 59, 136, 339, 340, 346) ont beaucoup trop de réponses manquantes sur les infos importantes : âge, race, domaine d'études, importance de la race/religion, ... .

In [11]:
# suppression des users 28  58  59 136 339 340 346
ids_to_remove = [28, 58, 59, 136, 339, 340, 346]
data.drop(data[data["iid"].isin(ids_to_remove)].index, inplace=True)
data.drop(data[data["pid"].isin(ids_to_remove)].index, inplace=True)

In [12]:
# Questionnaire rempli a l'inscription

# attr1_1      0.942946
# sinc1_1      0.942946
# intel1_1     0.942946
# fun1_1       1.062306
# amb1_1       1.181666
# shar1_1      1.444259


In [13]:
# attr4_1     22.547147
# sinc4_1     22.547147
# intel4_1    22.547147
# fun4_1      22.547147
# amb4_1      22.547147
# shar4_1     22.809740



In [14]:
with pd.option_context(
    "display.max_columns", None, 
    "display.max_rows", None,):
    pourcentage_na = data.isna().mean() * 100
    print(f"Pourcentage de données manquantes pour chaque colonne :\n{pourcentage_na}")

Pourcentage de données manquantes pour chaque colonne :
Unnamed: 0      0.000000
iid             0.000000
id              0.000000
gender          0.000000
idg             0.000000
condtn          0.000000
wave            0.000000
round           0.000000
position        0.000000
order           0.000000
partner         0.000000
pid             0.000000
match           0.000000
int_corr        0.000000
samerace        0.000000
age_o           0.377588
race_o          0.000000
pf_o_att        0.000000
pf_o_sin        0.000000
pf_o_int        0.000000
pf_o_fun        0.109622
pf_o_amb        0.219245
pf_o_sha        0.487211
dec_o           0.000000
attr_o          2.277710
sinc_o          3.154689
intel_o         3.398295
fun_o           4.056029
amb_o           8.428745
shar_o         12.740560
like_o          2.728380
prob_o          3.532278
met_o           4.336175
age             0.377588
field           0.000000
field_cd        0.000000
race            0.000000
imprace         0.0

In [15]:
print(f"Taux de NA au time 1 : {(data.loc[:, 'age':'amb5_1'].isna().mean() * 100).mean()}" )
#print(f"Taux de NA au halfway : {(data.loc[:, 'attr1_s':'amb3_s'].isna().mean() * 100).mean()}" )
print(f"Taux de NA au time 2 : {(data.loc[:, 'satis_2':'amb5_2'].isna().mean() * 100).mean()}" )
print(f"Taux de NA au time 3 : {(data.loc[:, 'you_call':].isna().mean() * 100).mean()}" )

Taux de NA au time 1 : 5.804527699609392
Taux de NA au time 2 : 32.71883332784673
Taux de NA au time 3 : 62.882460414129106


Au début de l'expérience, les gens remplissent bien les questionnaires (âge, études, préférences).  
Après l'événement (le lendemain et 3 semaines après), beaucoup abandonnent : 30-60% de cases vides pour les questions de suivi.  
Cette décroissance s'explique par le désintérêt progressif pour les questionnaires.

In [16]:
data.to_csv("../data/data_cleaned.csv")